## Overview

- Encrypting your passwords with Fernet
- Rotating your Fernet key
- Hiding variables from the UI
- Password authentication and DAGs filtering
- RBAC UI

## Securing Airflow

Airflow can be insecure especially if left unencrypted. The Postgres db used by Airflow does not encrypt password and other sensitive columns by default. This can be verified by going into the Postgres container and into the airflow db, check tables such as connection.

When you want to encrypt your sensitive data, the first thing you have to do is to install the package crypto along with airflow.
So, from your code editor, open the Dockerfile and add the package ‘crypto’ at the instruction where the pip install airflow is done.

Add a fernet key using the fernet_key variable in airflow.cfg file.

Fernet is a symmetric encryption method which makes sure that the value encrypted cannot be manipulated/read without the fernet key. This key is a URL-safe base64-encoded key with 32 bytes bringing the time when the value got encrypted. When a value needs to be encrypted, a fernet object is is instantiated based on that key and the method encrypt is called.

Go into the webserver or apiserver container and run bash
```bash
docker exec -it <apiserver_id> /bin/bash/

# Generate a Fernet object using Python cryptography module in the CLI
python -c "from cryptography.fernet import Fernet; print(Fernet.generate_key().decode())"
```
Copy the value and paste it as the value to fernet_key.

Note: Having the AIRFLOW__CORE__FERNET_KEY variable in docker-compose.yaml will overwrite the fernet_key in airflow.cfg  file.

#### Rotating the Fernet Key
Keep in mind is that changing your old key by a new one directly from the airflow.cfg file will cause decryption of existing credentials to fail.
There is a little process to follow in order to rotate the fernet without invalidating existing encrypted values.

Steps:
1. Set fernet_key to new_fernet_key,old_fernet_key
1. Run airflow rotate-fernet-key to re-encrypt existing credentials with the new fernet key
1. Set fernet_key to new_fernet_key

After successful rotation, restart the webserver/apiserver.

As a best practice, avoid using fernet_key in airflow.cfg as this is usually pushed into the remote repo. Instead use AIRFLOW__CORE__FERNET_KEY in docker-compose.yaml to generate the Fernet key as an environment variable.

#### Hiding Airflow Variables (in the UI)
By default hide_sensitive_var_conn_fields parameter in airflow.cfg is set to true. However, only variables with certain keywords are hidden or obfuscated in the UI such as those with "password". Here is a list of default sensitive keywords used by Airflow.
![](assets/default_sensitive_vars.jpg)

Also when fernet_key is used, new variables will be encrypted but those already existing will not.

Note: Variables can still be show in the logs.

Authentication to the Airflow UI can be set using auth_backends variable in airflow.cfg. 
Airflow Users can be created using Python. But first, you need to install the flask-bcrypt.
Run the script below in the webserver/apiserver container.

```python
import airflow
from airflow import models, settings
from airflow.contrib.auth.backends.password_auth import PasswordUser
user = PasswordUser(models.User())
user.username = 'admin'
user.email = 'admin@airflow.com'
user.password = 'admin'
session = settings.Session()
session.add(user)
session.commit()
session.close()
exit()

```